# Spark - PostgreSQL connectivity

## Introduction
This notebook will be used to check if connection among Spark and 
postgreSQL database is working. 

## Utils

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder 
    .appName("spark-postgresql-connectivity") 
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate()
)

In [ ]:
import os 
# TODO update to be able of loading credentials from env vars 

user = os.getenv('POSTGRES_USER')
psswd = os.getenv('POSTGRES_PASSWORD')

host = os.getenv('POSTGRES_HOST')
port = os.getenv('POSTGRES_PORT')
db = os.getenv('POSTGRES_DB')
jdbc_url = f"jdbc:postgresql://{host}:{port}/{db}"

## Validations

### Reading

In [ ]:
df = (
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("url", jdbc_url)
    .option("user", user)
    .option("password", psswd)
    .option("query", "select 1 as select_test")
    .load()
)

df.show()

### Writing

In [ ]:
# Sample data
data = [
    (1, "Alice", 30),
    (2, "Bob", 25),
    (3, "Charlie", 35)
]
columns = ["id", "name", "age"]

df = spark.createDataFrame(data, columns)

# PostgreSQL connection settings
properties = {
    "user": user,
    "password": psswd,
    "driver": "org.postgresql.Driver"
}

# Write DataFrame to PostgreSQL table
df.write.jdbc(
    url=jdbc_url,
    table="public.people",   # schema.table
    mode="overwrite",        # overwrite, append, ignore, error
    properties=properties
)

In [ ]:
(
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("url", jdbc_url)
    .option("user", user)
    .option("password", psswd)
    .option("dbtable", "public.people")
    .load()
).show()

In [ ]:
(
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("url", jdbc_url)
    .option("user", user)
    .option("password", psswd)
    .option("query", "select * from public.people")
    .load()
).show()

### Drop 

In [ ]:
# TODO jdbc connection cannot handle this, implement a way using sql_alchemy or other library